In [41]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

In [42]:
protein = "VCAM1"
synapse_type = "VGLUT1-PSD95"

In [73]:
results_file = f"/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/{protein}/{protein}-LacZ_{synapse_type}_output_data/metric_results.csv"
results = pd.read_csv(results_file)

In [74]:
# Add a new column 'section' by extracting the "section-<number>" part from 'img_filename'
results['section'] = results['img_filename'].apply(lambda x: re.search(r'section-\d+', x).group(0) if re.search(r'section-\d+', x) else None)

# Select the relevant columns
results_mfi = results[['presynapse_image_mfi', 'gRNA', 'hippocampal_layer', 'section', 'Brain']]

In [75]:
scaler = StandardScaler()
results_mfi['presynapse_image_mfi_scaled'] = scaler.fit_transform(results_mfi[['presynapse_image_mfi']])

/var/folders/p5/hzbdgkws2lvg79nyhlq_g_sm0000gn/T/ipykernel_86018/3161662538.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_mfi['presynapse_image_mfi_scaled'] = scaler.fit_transform(results_mfi[['presynapse_image_mfi']])


In [78]:
results_mfi.head(10)

,presynapse_image_mfi,gRNA,hippocampal_layer,section,Brain,presynapse_image_mfi_scaled,Brain_gRNA
0,442.3194247,VCAM1-gRNA,DG Hilus,section-1,Brain-7,-1.1613352,Brain-7:VCAM1-gRNA
1,1589.7434762,LacZ-gRNA,CA3 SO,section-2,Brain-5,2.6986663,Brain-5:LacZ-gRNA
2,742.1220367,VCAM1-gRNA,CA1 SO,section-2,Brain-4,-0.1527817,Brain-4:VCAM1-gRNA
3,493.6974500,VCAM1-gRNA,CA3 SL,section-3,Brain-4-2,-0.9884965,Brain-4-2:VCAM1-gRNA
4,857.1568904,LacZ-gRNA,DG Hilus,section-3,Brain-5,0.2342023,Brain-5:LacZ-gRNA
5,899.5746396,VCAM1-gRNA,DG ML,section-1,Brain-5,0.3768980,Brain-5:VCAM1-gRNA
6,835.5293111,LacZ-gRNA,CA3 SR,section-1,Brain-4,0.1614458,Brain-4:LacZ-gRNA
7,1150.6024550,LacZ-gRNA,CA3 SO,section-3,Brain-5,1.2213703,Brain-5:LacZ-gRNA
8,1061.3039368,LacZ-gRNA,CA1 SO,section-1,Brain-4,0.9209648,Brain-4:LacZ-gRNA
9,594.6287821,VCAM1-gRNA,DG Hilus,section-2,Brain-7,-0.6489576,Brain-7:VCAM1-gRNA


In [80]:
# Create a combined key for the nested random effects as needed
results_mfi['Brain_gRNA'] = results_mfi['Brain'] + ':' + results_mfi['gRNA']
results_mfi['Brain_section_layer'] = results_mfi['Brain'] + ':' + results_mfi['section'] + ':' + results_mfi['hippocampal_layer']

/var/folders/p5/hzbdgkws2lvg79nyhlq_g_sm0000gn/T/ipykernel_86018/1634473522.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_mfi['Brain_gRNA'] = results_mfi['Brain'] + ':' + results_mfi['gRNA']
/var/folders/p5/hzbdgkws2lvg79nyhlq_g_sm0000gn/T/ipykernel_86018/1634473522.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_mfi['Brain_section_layer'] = results_mfi['Brain'] + ':' + results_mfi['section'] + ':' + results_mfi['hippocampal_layer']


In [81]:
results_mfi.head(10)

,presynapse_image_mfi,gRNA,hippocampal_layer,section,Brain,presynapse_image_mfi_scaled,Brain_gRNA,Brain_section_layer
0,442.3194247,VCAM1-gRNA,DG Hilus,section-1,Brain-7,-1.1613352,Brain-7:VCAM1-gRNA,Brain-7:section-1:DG Hilus
1,1589.7434762,LacZ-gRNA,CA3 SO,section-2,Brain-5,2.6986663,Brain-5:LacZ-gRNA,Brain-5:section-2:CA3 SO
2,742.1220367,VCAM1-gRNA,CA1 SO,section-2,Brain-4,-0.1527817,Brain-4:VCAM1-gRNA,Brain-4:section-2:CA1 SO
3,493.6974500,VCAM1-gRNA,CA3 SL,section-3,Brain-4-2,-0.9884965,Brain-4-2:VCAM1-gRNA,Brain-4-2:section-3:CA3 SL
4,857.1568904,LacZ-gRNA,DG Hilus,section-3,Brain-5,0.2342023,Brain-5:LacZ-gRNA,Brain-5:section-3:DG Hilus
5,899.5746396,VCAM1-gRNA,DG ML,section-1,Brain-5,0.3768980,Brain-5:VCAM1-gRNA,Brain-5:section-1:DG ML
6,835.5293111,LacZ-gRNA,CA3 SR,section-1,Brain-4,0.1614458,Brain-4:LacZ-gRNA,Brain-4:section-1:CA3 SR
7,1150.6024550,LacZ-gRNA,CA3 SO,section-3,Brain-5,1.2213703,Brain-5:LacZ-gRNA,Brain-5:section-3:CA3 SO
8,1061.3039368,LacZ-gRNA,CA1 SO,section-1,Brain-4,0.9209648,Brain-4:LacZ-gRNA,Brain-4:section-1:CA1 SO
9,594.6287821,VCAM1-gRNA,DG Hilus,section-2,Brain-7,-0.6489576,Brain-7:VCAM1-gRNA,Brain-7:section-2:DG Hilus


In [86]:
# for decimals
pd.set_option('display.precision', 7)

# Define the mixed-effects model
model = smf.mixedlm(
    "presynapse_image_mfi_scaled ~ gRNA * hippocampal_layer",   # Fixed effects formula
    data=results_mfi,
    groups="Brain",                                            # Random intercept for Brain
    re_formula="~1",                                           # Include random intercepts
    vc_formula={
        "Brain_gRNA": "0 + Brain_gRNA",                        # Paired design of hemispheres
        "Brain_section_layer": "0 + Brain_section_layer"       # Nested measurements
    }
)

# Fit the model using REML
model_results = model.fit(reml=True)

# Print the summary of the model
print(model_results.summary())

# To get the variance components
print("\nVariance Components:")
print(model_results.cov_re)

                           Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       presynapse_image_mfi_scaled
No. Observations:       176           Method:                   REML                       
No. Groups:             4             Scale:                    0.1698                     
Min. group size:        32            Log-Likelihood:           -137.2947                  
Max. group size:        48            Converged:                Yes                        
Mean group size:        44.0                                                               
-------------------------------------------------------------------------------------------
                                                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------------------------
Intercept                                        -1.153    0.234 -4.936 0.000 -1.610 -0.695
gRNA[T.VCAM1-gR

/Users/cgeyskens/miniconda3/envs/synapse-counting-2/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [84]:
# Get the summary of the results
summary = model_results.summary()

# Extract coefficients and p-values into a DataFrame
coefficients = model_results.params
p_values = model_results.pvalues

adj_pvals = multipletests(p_values, method='fdr_bh')[1]
significance = ['Significant' if p < 0.05 else 'Not Significant' for p in adj_pvals]


# Create a DataFrame for coefficients and p-values
coefficients_table = pd.DataFrame({
    'Variable': coefficients.index,
    'Coefficient': coefficients,
    'P > z': p_values,
    "adj_pvals": adj_pvals,
    "significance": significance
})

# Reset index for better formatting (optional)
coefficients_table.reset_index(drop=True, inplace=True)

# Display the coefficients table
coefficients_table.head(20)

,Variable,Coefficient,P > z,adj_pvals,significance
0,Intercept,-1.1525867,7.9757980e-07,1.8942520e-06,Significant
1,gRNA[T.VCAM1-gRNA],0.0105405,9.5356977e-01,9.5356977e-01,Not Significant
2,hippocampal_layer[T.CA1 SO],1.7886114,1.2114634e-16,1.1508902e-15,Significant
3,hippocampal_layer[T.CA1 SR],1.0973454,3.7517086e-07,1.0183209e-06,Significant
4,hippocampal_layer[T.CA3 SL],1.5665134,4.0590939e-13,2.5707595e-12,Significant
5,hippocampal_layer[T.CA3 SO],3.3714960,6.0925086e-55,1.1575766e-53,Significant
6,hippocampal_layer[T.CA3 SR],1.3904966,1.2062118e-10,5.7295062e-10,Significant
7,hippocampal_layer[T.DG Hilus],0.9678248,7.4151026e-06,1.5654105e-05,Significant
8,hippocampal_layer[T.DG ML],1.1398236,1.3071984e-07,4.9673539e-07,Significant
9,gRNA[T.VCAM1-gRNA]:hippocampal_layer[T.CA1 SO],-0.2836247,2.5363945e-01,3.1264555e-01,Not Significant
